# ⚔️ Red Neuronal con Backpropagation — Desde Cero
**Módulo 4 — Diplomado Deep Learning | Diana Blanco**

---

## ¿Qué hace este notebook?

Hasta ahora habíamos construido el *forward pass* a mano — la red predecía, pero no aprendía.
Este notebook agrega lo que faltaba: **backpropagation** — el algoritmo que permite a la red *aprender* de sus errores.

### La analogía del videojuego 🎮

> Imagina que entrenas a un personaje de RPG.  
> Cada vez que pierde un combate (hace una predicción mala), revisas *qué habilidades falló*,  
> y ajustas sus stats en la dirección correcta para que la próxima batalla le vaya mejor.  
> Eso es exactamente backpropagation: analizar el error, atribuirlo a cada peso, y ajustar.

### Mapa del notebook:

| Sección | Contenido |
|---|---|
| Parte 1 | Clase `Value` — cada número lleva su propio gradiente |
| Parte 2 | Operaciones diferenciables (`+`, `*`, `sigmoid`) |
| Parte 3 | Backpropagation con `backward()` |
| Parte 4 | Neurona, Capa y Red con entrenamiento |
| Demo 1 | Entrenamiento XOR — la red aprende de verdad |
| Demo 2 | Clasificador de círculo entrenado |

> 💡 **Diferencia clave con el notebook anterior:**  
> Antes los pesos estaban fijos (a mano o aleatorios). Ahora la red los **ajusta sola** después de cada predicción.

---
## 🔮 Parte 1: La clase `Value` — Números que recuerdan su historia

Este es el corazón de todo. En una red neuronal normal, los números son solo números.
Pero para que funcione el backpropagation, cada número necesita recordar:

1. **Su valor actual** (`data`) — el número en sí
2. **Su gradiente** (`grad`) — cuánto afecta al error final
3. **De dónde vino** (`_children`) — qué operación lo creó
4. **Cómo propagar el gradiente** (`_backward`) — la regla de la cadena específica para esa operación

### La analogía del árbol genealógico 🌳

> Cada `Value` conoce a sus "padres" (los valores que lo generaron).  
> Cuando calculamos el error final, le preguntamos a cada nodo cuánta responsabilidad tuvo,  
> y ese nodo les pasa la responsabilidad a sus padres — hasta llegar a los pesos originales.  
> Eso es la regla de la cadena aplicada automáticamente.

### ¿Por qué no usar NumPy o TensorFlow directamente?

Porque construirlo desde cero muestra *exactamente* qué hace PyTorch/TensorFlow por dentro.
Después de esto, `loss.backward()` en PyTorch no va a ser magia — vas a saber exactamente qué ocurre.

In [1]:
import math
import random

# ═══════════════════════════════════════════════════════════════════════════
# CLASE: Value
# Un número que recuerda de dónde vino y cómo contribuyó al resultado final.
# En PyTorch, esto equivale a un Tensor con requires_grad=True.
# ═══════════════════════════════════════════════════════════════════════════

class Value:

    def __init__(self, data, children=(), op=''):
        """
        data:     el valor numérico real
        children: los Values que generaron este (sus 'padres' en el grafo)
        op:       la operación que lo creó ('+', '*', 'sigmoid')

        Explicación de cada atributo:
          .data        → el número real (ej: 3.0, -2.5)
          .grad        → gradiente acumulado ∂Loss/∂(este_valor). Inicia en 0.0
                         y se va SUMANDO (no asignando) porque un valor puede
                         contribuir a múltiples pérdidas (ej: un peso usado en
                         múltiples ejemplos). En la clase se explica por qué +=.
          ._backward   → función que implementa la regla de la cadena para la
                         operación que creó este valor. Por defecto no hace nada
                         (lambda: None), y cada operación (suma, multi, sigmoid)
                         le asigna su propia versión.
          ._children   → conjunto de Values que fueron operandos para crear este.
                         Forman el grafo computacional hacia atrás.
          ._op         → solo para depuración: '+', '*', 'sigmoid'
        """
        self.data = data
        self.grad = 0.0          # gradiente inicial = 0 (se acumula durante backward)
        self._backward = lambda: None   # función de retropropagación (vacía por defecto)
        self._children = set(children) # quiénes generaron este valor
        self._op = op

    def __repr__(self):
        # Representación al imprimir: muestra valor y gradiente actual
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"


# ── Demostración: crear valores y ver su estructura ──────────────────────────
a = Value(3.0)
b = Value(2.0)
print(f"a = {a}")
print(f"b = {b}")
print(f"a.data = {a.data}, a.grad = {a.grad} (gradiente en 0 hasta hacer backward)")
print(f"a._children = {a._children} (vacío: no tiene padres, es un valor base)")

a = Value(data=3.0000, grad=0.0000)
b = Value(data=2.0000, grad=0.0000)
a.data = 3.0, a.grad = 0.0 (gradiente en 0 hasta hacer backward)
a._children = set() (vacío: no tiene padres, es un valor base)


---
## ➕✖️ Parte 2: Operaciones diferenciables

Para que backpropagation funcione, **cada operación matemática** debe saber cómo
distribuir el gradiente hacia atrás — eso es la regla de la cadena.

### La regla de la cadena en términos simples 🎯

> Si el costo C depende de z, y z depende de w:
> $$\frac{\partial C}{\partial w} = \frac{\partial C}{\partial z} \cdot \frac{\partial z}{\partial w}$$
> Cada operación sabe su pedacito de la cadena. Al encadenarlas todas, obtenemos el gradiente total.

### Las tres operaciones que necesitamos:

| Operación | Forward | Backward (regla de la cadena) |
|---|---|---|
| `a + b = out` | `out.data = a.data + b.data` | `a.grad += out.grad · 1`, `b.grad += out.grad · 1` |
| `a * b = out` | `out.data = a.data * b.data` | `a.grad += out.grad · b.data`, `b.grad += out.grad · a.data` |
| `sigmoid(a) = out` | `out.data = σ(a.data)` | `a.grad += out.grad · σ(a) · (1 - σ(a))` |

La derivada de sigmoid es especialmente bonita: `σ'(z) = σ(z) · (1 - σ(z))`  
Ya calculamos sigmoid en el forward, así que reusar ese valor es gratuito. 🎁

In [2]:
# ── Continuación de la clase Value: agregamos las operaciones ────────────────
# (Redefinimos la clase completa para que sea autocontenida)

class Value:
    def __init__(self, data, children=(), op=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._children = set(children)
        self._op = op

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    # ── SUMA: a + b ──────────────────────────────────────────────────────────
    def __add__(self, other):
        # Si 'other' es un número normal (int/float), lo envolvemos en Value
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            # Regla de la cadena para suma:
            #   Fórmula: out = a + b
            #   ∂out/∂a = 1,  ∂out/∂b = 1
            #   El gradiente fluye COMPLETO e IGUAL a ambos operandos
            #   (la suma reparte el gradiente sin modificarlo)
            #
            # Usamos += en vez de = porque un mismo valor puede participar
            # en MÚLTIPLES operaciones. Sus gradientes se SUMAN (por la regla
            # de la suma en cálculo: si f = g + h, entonces f' = g' + h').
            self.grad  += out.grad
            other.grad += out.grad

        out._backward = _backward
        return out

    def __radd__(self, other):
        # Permite '3 + Value(2)' además de 'Value(2) + 3'
        return self.__add__(other)

    # ── MULTIPLICACIÓN: a * b ────────────────────────────────────────────────
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            # Regla de la cadena para multiplicación:
            #   Fórmula: out = a * b
            #   ∂out/∂a = b,  ∂out/∂b = a
            #   El gradiente de a = (valor de b) × (gradiente de salida)
            #   y viceversa — porque la derivada de a*b respecto a 'a' es 'b'
            self.grad  += other.data * out.grad
            other.grad += self.data  * out.grad

        out._backward = _backward
        return out

    def __rmul__(self, other):
        return self.__mul__(other)

    # ── NEGACIÓN y RESTA (construidas sobre * y +) ───────────────────────────
    def __neg__(self):
        # -a = a * (-1)  → reutiliza __mul__ con -1
        return self * -1

    def __sub__(self, other):
        # a - b = a + (-b)  → reutiliza suma y negación
        return self + (-other)

    # ── SIGMOID ──────────────────────────────────────────────────────────────
    def sigmoid(self):
        x = max(-500, min(500, self.data))  # clamp anti-overflow en la entrada
        s = 1.0 / (1.0 + math.exp(-x))     # σ(z) = 1 / (1 + e^(-z))
        out = Value(s, (self,), 'sigmoid')

        def _backward():
            # Derivada de sigmoid:
            #   Fórmula: σ'(z) = σ(z) · (1 - σ(z))
            #   Ya calculamos σ(z) = 's' en el forward pass (línea arriba)
            #   Lo reutilizamos aquí → ahorro computacional
            #   Ventaja: sigmoid es la ÚNICA función cuya derivada se
            #   expresa en términos de sí misma, sin recalcular.
            self.grad += (s * (1 - s)) * out.grad

        out._backward = _backward
        return out


    # ── BACKWARD: iniciar backpropagation desde este nodo ──────────────────
    def backward(self):
        """
        Ejecuta backpropagation: calcula dLoss/d(v) para cada Value v
        en el grafo computacional desde este nodo hacia las hojas.
        Equivalente a: loss.backward() en PyTorch.
        """
        # 1. Construir orden topologico (padres antes que hijos)
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._children:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        # 2. dloss/dloss = 1 (punto de partida)
        self.grad = 1.0
        # 3. Propagar gradientes en orden inverso: salida -> entradas
        for v in reversed(topo):
            v._backward()

# ── Verificación manual de la suma ───────────────────────────────────────────
a = Value(3.0)
b = Value(2.0)
c = a + b          # forward: c.data = 5.0
print(f"a + b = {c}  ← gradientes en 0 todavía")

# Simulamos que el gradiente del resultado es 1.0 (como si c fuera la salida final)
c.grad = 1.0
c._backward()      # backward: distribuye el gradiente hacia a y b
print(f"Después de backward:")
print(f"  a.grad = {a.grad}  ← ∂c/∂a = 1 (la suma le pasa el gradiente sin cambios)")
print(f"  b.grad = {b.grad}  ← ∂c/∂b = 1")
print()

# Verificación de la multiplicación
x = Value(4.0)
y = Value(3.0)
z = x * y          # z = 12
z.grad = 1.0
z._backward()
print(f"x * y = {z}")
print(f"  x.grad = {x.grad}  ← ∂z/∂x = y = 3.0")
print(f"  y.grad = {y.grad}  ← ∂z/∂y = x = 4.0")

a + b = Value(data=5.0000, grad=0.0000)  ← gradientes en 0 todavía
Después de backward:
  a.grad = 1.0  ← ∂c/∂a = 1 (la suma le pasa el gradiente sin cambios)
  b.grad = 1.0  ← ∂c/∂b = 1

x * y = Value(data=12.0000, grad=1.0000)
  x.grad = 3.0  ← ∂z/∂x = y = 3.0
  y.grad = 4.0  ← ∂z/∂y = x = 4.0


---
## 🔄 Parte 3: `backward()` — Retropropagación automática

El método `backward()` recorre el grafo computacional **de atrás hacia adelante**
y llama a `_backward()` en cada nodo para distribuir los gradientes.

### ¿Qué es el 'orden topológico'? 🗺️

> Imagina que tienes una cadena de producción en un RPG:
> Madera + Hierro → Espada → Ataque
> Para calcular cuánto contribuyó la Madera al daño total,
> necesitas procesar primero el Ataque, luego la Espada, y al final los ingredientes.
> Orden topológico = de los efectos hacia las causas.

### El algoritmo:
1. Construir la lista de nodos en orden topológico (de hojas a raíz)
2. Fijar el gradiente de la salida en 1.0 (∂salida/∂salida = 1)
3. Recorrer la lista **al revés** (de raíz a hojas) llamando `_backward()` en cada nodo

In [3]:
# Ejemplo: calcular gradientes en expresion simple
# loss = (w * x + b).sigmoid()
# Queremos: dloss/dw, dloss/dx, dloss/db

# NOTA: la clase Value YA tiene el metodo backward() incluido arriba.
# Creamos valores base (w, x, b) y hacemos operaciones para construir el grafo.
w = Value(0.5)
x = Value(2.0)
b = Value(0.1)

z   = w * x + b          # suma ponderada: z = w*x + b
out = z.sigmoid()        # activacion: a = sigmoid(z)
out.backward()           # backprop: calcula dout/dw, dout/dx, dout/db

print("Ejemplo: out = sigmoid(w*x + b)")
print(f"  w={w.data}, x={x.data}, b={b.data}")
print(f"  z = w*x+b = {z.data:.4f}")
print(f"  out = sigmoid(z) = {out.data:.4f}")
print()
print("Gradientes calculados por backward():")
print(f"  dout/dw = {w.grad:.4f}")
print(f"  dout/dx = {x.grad:.4f}")
print(f"  dout/db = {b.grad:.4f}")
print()
print("Gradiente positivo -> aumentar parametro aumenta perdida.")
print("El optimizador RESTA grad*lr -> cuesta abajo hacia el minimo.")


Ejemplo: out = sigmoid(w*x + b)
  w=0.5, x=2.0, b=0.1
  z = w*x+b = 1.1000
  out = sigmoid(z) = 0.7503

Gradientes calculados por backward():
  dout/dw = 0.3747
  dout/dx = 0.0937
  dout/db = 0.1874

Gradiente positivo -> aumentar parametro aumenta perdida.
El optimizador RESTA grad*lr -> cuesta abajo hacia el minimo.


---
## 🧩 Parte 4: Neurona, Capa y Red — Con entrenamiento

Ahora sí construimos la red con todo el equipo:

- `Neuron`: usa `Value` en cada peso → los gradientes fluyen automáticamente
- `Layer`: grupo de neuronas
- `Network`: capas encadenadas + `zero_grad()` para limpiar entre epochs

### Diferencias clave vs. el notebook anterior 🆚

| Antes (Red_neuronal.ipynb) | Este notebook |
|---|---|
| Pesos eran `float` normales | Pesos son `Value` con `.grad` |
| Solo forward pass | Forward + backward completo |
| Sin entrenamiento | Aprende sola con gradiente descendente |
| Necesitaba pesos a mano para XOR | Encuentra los pesos sola en ~1000 epochs |

### Inicialización He (Xavier simplificado) 🎲

Los pesos no se inicializan en rango [-1,1] arbitrario — se usa una escala calculada:
```
scale = sqrt(2 / n_inputs)
```
Esto evita que la red empiece con gradientes demasiado grandes o pequeños.
Es la misma inicialización que PyTorch usa por defecto en `nn.Linear`.

In [4]:
# ═══════════════════════════════════════════════════════════════════════════
# CLASE: Neuron — Una neurona con pesos entrenables
# ═══════════════════════════════════════════════════════════════════════════

class Neuron:
    def __init__(self, n_inputs):
        # Inicialización He: scale = sqrt(2/n_inputs)
        # Evita que los gradientes exploten o se desvanezcan al inicio
        scale = (2.0 / n_inputs) ** 0.5
        # Cada peso es un Value — así el gradiente fluye a través de él
        self.weights = [Value(random.uniform(-scale, scale)) for _ in range(n_inputs)]
        self.bias    = Value(0.0)  # sesgo iniciado en 0 (convención estándar)

    def __call__(self, x):
        """
        Forward pass de la neurona:
        1. Suma ponderada: sum(w_i * x_i) + bias
        2. Activación sigmoid
        """
        # sum(..., start) acumula todos los w*x partiendo desde self.bias
        # Equivalente a: act = bias + w0*x0 + w1*x1 + ...
        act = sum((wi * xi for wi, xi in zip(self.weights, x)), self.bias)
        return act.sigmoid()

    def parameters(self):
        # Lista de todos los parámetros entrenables: pesos + sesgo
        # Usada para iterar y actualizar con gradiente descendente
        return self.weights + [self.bias]


# ═══════════════════════════════════════════════════════════════════════════
# CLASE: Layer — Grupo de neuronas en paralelo
# ═══════════════════════════════════════════════════════════════════════════

class Layer:
    def __init__(self, n_inputs, n_outputs):
        self.neurons = [Neuron(n_inputs) for _ in range(n_outputs)]

    def __call__(self, x):
        # Cada neurona procesa las MISMAS entradas independientemente
        out = [n(x) for n in self.neurons]
        # Si solo hay 1 neurona, devolver el valor directamente (no una lista)
        # Esto simplifica el código para la capa de salida binaria
        return out[0] if len(out) == 1 else out

    def parameters(self):
        # Recolectar parámetros de TODAS las neuronas de esta capa
        params = []
        for n in self.neurons:
            params.extend(n.parameters())
        return params


# ═══════════════════════════════════════════════════════════════════════════
# CLASE: Network — Capas encadenadas con entrenamiento
# ═══════════════════════════════════════════════════════════════════════════

class Network:
    def __init__(self, sizes):
        """
        sizes: lista de tamaños [entrada, oculta1, ..., salida]
        Ejemplo: [2, 4, 1] crea una red 2→4→1
        Las capas se construyen automáticamente como pares consecutivos.
        """
        self.layers = []
        for i in range(len(sizes) - 1):
            self.layers.append(Layer(sizes[i], sizes[i + 1]))

    def __call__(self, x):
        # Forward pass completo: propagar por todas las capas en orden
        for layer in self.layers:
            x = layer(x)
            # Normalizar: si la capa devuelve un Value (no lista), envolverlo
            if not isinstance(x, list):
                x = [x]
        # Si la salida es un solo valor, devolver el Value directamente
        return x[0] if len(x) == 1 else x

    def parameters(self):
        # Todos los parámetros de toda la red (para el optimizador)
        params = []
        for layer in self.layers:
            params.extend(layer.parameters())
        return params

    def zero_grad(self):
        """
        Resetea todos los gradientes a 0 antes de cada paso de entrenamiento.
        CRUCIAL: sin esto los gradientes se acumulan entre epochs y el
        entrenamiento diverge. En PyTorch es optimizer.zero_grad().
        """
        for p in self.parameters():
            p.grad = 0.0


print("Clases Neuron, Layer y Network definidas ✓")
red_test = Network([2, 4, 1])
print(f"Red 2-4-1: {len(red_test.parameters())} parámetros entrenables")
print(f"  (2×4 + 4 sesgos) + (4×1 + 1 sesgo) = 12 + 5 = 17 parámetros")

Clases Neuron, Layer y Network definidas ✓
Red 2-4-1: 17 parámetros entrenables
  (2×4 + 4 sesgos) + (4×1 + 1 sesgo) = 12 + 5 = 17 parámetros


---
## 🔧 Función de pérdida: MSE

Para entrenar la red necesitamos medir qué tan equivocada está la predicción.
Usamos **Error Cuadrático Medio (MSE)** para un solo ejemplo:

$$\text{MSE} = (\hat{y} - y)^2$$

¿Por qué al cuadrado? Dos razones:
1. Siempre positivo (el error no puede cancelarse)
2. Penaliza más los errores grandes (2 de error → 4 de pérdida; 1 de error → 1 de pérdida)

Y como ya tenemos `__sub__` y `__mul__` en `Value`, esto se construye solo — y el gradiente fluye automáticamente.

In [5]:
def mse_loss(predicted, target):
    """
    Error Cuadrático para un solo ejemplo: (predicción - valor_real)²
    
    Construido sobre las operaciones de Value, así el gradiente
    fluye automáticamente hasta los pesos cuando llamamos .backward()
    
    predicted: Value (salida de la red)
    target:    float (valor real que esperamos)
    """
    diff = predicted + Value(-target)   # predicción - target (resta)
    return diff * diff                  # al cuadrado


# ── Verificar que MSE produce gradientes correctos ───────────────────────────
pred_test = Value(0.8)    # la red predijo 0.8
target_test = 1.0         # el valor real era 1.0
loss_test = mse_loss(pred_test, target_test)
loss_test.backward()

print(f"Predicción: {pred_test.data}, Real: {target_test}")
print(f"MSE = ({pred_test.data} - {target_test})² = {loss_test.data:.4f}")
print(f"∂MSE/∂pred = {pred_test.grad:.4f}  ← negativo: la pred era MUY BAJA")
print(f"  El optimizador aumentará la predicción (resta gradiente negativo)")

Predicción: 0.8, Real: 1.0
MSE = (0.8 - 1.0)² = 0.0400
∂MSE/∂pred = -0.4000  ← negativo: la pred era MUY BAJA
  El optimizador aumentará la predicción (resta gradiente negativo)


---
## 🎯 Demo 1: Entrenar XOR — La red aprende de verdad

Ahora sí entrenamos. La diferencia con el notebook anterior es enorme:
antes los pesos XOR los pusimos a mano. Ahora la red los encuentra **sola**.

### El ciclo de entrenamiento (loop de entrenamiento) ⚔️

```
Para cada epoch:
  1. Forward pass → calcular pérdida total sobre todos los datos
  2. zero_grad()  → limpiar gradientes del paso anterior
  3. backward()   → calcular gradientes nuevos (backprop)
  4. Actualizar   → w = w - lr × ∂Loss/∂w  (descenso por gradiente)
```

Esto es exactamente lo que hace `optimizer.step()` en PyTorch — pero aquí lo vemos línea por línea.

In [6]:
def train_xor():
    print("=" * 52)
    print("Entrenamiento XOR - red 2-4-1")
    print("=" * 52)

    random.seed(42)   # reproducibilidad
    net = Network([2, 4, 1])   # arquitectura: 2 entradas, 4 ocultas, 1 salida

    xor_data = [
        ([0.0, 0.0], 0.0),
        ([0.0, 1.0], 1.0),
        ([1.0, 0.0], 1.0),
        ([1.0, 1.0], 0.0),
    ]

    learning_rate = 1.0   # paso de actualizacion

    for epoch in range(1000):

        # PASO 1: Forward pass - predecir y acumular perdida
        # Formula: z = w*a_prev + b,  a = sigmoid(z),  C = sum((a-y)^2)
        total_loss = Value(0.0)
        for inputs, target in xor_data:
            x    = [Value(i) for i in inputs]
            pred = net(x)                     # a(L) = prediccion
            loss = mse_loss(pred, target)     # C = (a(L) - y)^2
            total_loss = total_loss + loss    # sumar sobre todos los ejemplos

        # PASO 2: zero_grad - limpiar gradientes del epoch anterior
        # Equivalente a: optimizer.zero_grad() en PyTorch
        net.zero_grad()

        # PASO 3: Backpropagation - regla de la cadena
        # Formula: dC/dw = a_prev * sigmoid_prime(z) * 2*(a - y)
        total_loss.backward()

        # PASO 4: Descenso por gradiente - actualizar pesos
        # Formula: w_nuevo = w_viejo - lr * dC/dw
        # Si gradiente > 0: perdida sube al aumentar w, entonces BAJAMOS w
        # Si gradiente < 0: perdida baja al aumentar w, entonces SUBIMOS w
        for p in net.parameters():
            p.data -= learning_rate * p.grad

        if epoch % 100 == 0:
            print(f"  Epoch {epoch:4d} | Loss: {total_loss.data:.6f}")

    # Resultados finales
    print("\nResultados XOR tras el entrenamiento:")
    for inputs, target in xor_data:
        x    = [Value(i) for i in inputs]
        pred = net(x)
        cl   = 1 if pred.data > 0.5 else 0
        ok   = "OK" if cl == int(target) else "X"
        print(f"  {inputs} -> {pred.data:.4f} -> clase {cl} (esperado: {int(target)}) {ok}")


train_xor()


Entrenamiento XOR - red 2-4-1
  Epoch    0 | Loss: 1.067385
  Epoch  100 | Loss: 0.805927
  Epoch  200 | Loss: 0.190474
  Epoch  300 | Loss: 0.035562
  Epoch  400 | Loss: 0.015281
  Epoch  500 | Loss: 0.009271
  Epoch  600 | Loss: 0.006544
  Epoch  700 | Loss: 0.005017
  Epoch  800 | Loss: 0.004050
  Epoch  900 | Loss: 0.003385

Resultados XOR tras el entrenamiento:
  [0.0, 0.0] -> 0.0251 -> clase 0 (esperado: 0) OK
  [0.0, 1.0] -> 0.9677 -> clase 1 (esperado: 1) OK
  [1.0, 0.0] -> 0.9796 -> clase 1 (esperado: 1) OK
  [1.0, 1.0] -> 0.0286 -> clase 0 (esperado: 0) OK


---
## ⭕ Demo 2: Clasificador de círculo con entrenamiento real

En el notebook anterior la red con el círculo tenía ~17% de accuracy porque los pesos eran aleatorios.
Ahora entrenamos de verdad y llegamos a 100%.

### Diferencias de estrategia vs. XOR:

| Aspecto | XOR | Círculo |
|---|---|---|
| Datos | 4 ejemplos fijos | 80 generados aleatoriamente |
| Batch | Todos a la vez (batch GD) | Uno por uno (SGD estocástico) |
| Learning rate | 1.0 | 0.5 (más pequeño por SGD) |
| Parada | 1000 epochs fijas | Para cuando llega a 100% |

**SGD estocástico (un ejemplo a la vez):** actualiza los pesos más seguido pero con más ruido.
Para el círculo funciona mejor porque los 80 puntos tienen variedad y el ruido ayuda a escapar de mínimos locales.

In [7]:
import matplotlib.pyplot as plt

def generate_circle_data(n=100):
    """Genera n puntos en [-1.5, 1.5]^2 con label=1 si dentro del circulo radio=1."""
    data = []
    for _ in range(n):
        x1 = random.uniform(-1.5, 1.5)
        x2 = random.uniform(-1.5, 1.5)
        label = 1.0 if x1*x1 + x2*x2 < 1.0 else 0.0
        data.append(([x1, x2], label))
    return data


def train_circle():
    print("=" * 52)
    print("Entrenamiento - Clasificador de Circulo")
    print("=" * 52)

    random.seed(7)
    circle_data   = generate_circle_data(80)
    net           = Network([2, 8, 1])
    learning_rate = 0.5

    for epoch in range(200):
        # Shuffle cada epoch para evitar memorizar el orden
        random.shuffle(circle_data)
        total_loss_val = 0.0

        for inputs, target in circle_data:
            # SGD: actualizar pesos tras CADA ejemplo (no batch completo)
            x    = [Value(i) for i in inputs]
            pred = net(x)
            loss = mse_loss(pred, target)

            net.zero_grad()
            loss.backward()
            for p in net.parameters():
                p.data -= learning_rate * p.grad

            total_loss_val += loss.data

        if epoch % 10 == 0:
            correct = sum(
                1 for inputs, target in circle_data
                if (1.0 if net([Value(i) for i in inputs]).data > 0.5 else 0.0) == target
            )
            accuracy = correct / len(circle_data) * 100
            print(f"  Epoch {epoch:4d} | Loss: {total_loss_val:.4f} | Accuracy: {accuracy:.1f}%")
            if accuracy == 100:
                print("  100% alcanzado! Deteniendo entrenamiento.")
                break

    print("\nResultados en puntos de prueba:")
    test_points = [
        ([0.0, 0.0],  "dentro"),
        ([0.5, 0.5],  "dentro"),
        ([1.2, 1.2],  "fuera"),
        ([0.0, 1.2],  "fuera"),
        ([-0.3, 0.3], "dentro"),
    ]
    for point, expected in test_points:
        pred = net([Value(i) for i in point])
        predicted = "dentro" if pred.data > 0.5 else "fuera"
        ok = "OK" if predicted == expected else "X"
        print(f"  {point} -> {pred.data:.4f} ({predicted}, esperado {expected}) {ok}")


train_circle()


Entrenamiento - Clasificador de Circulo
  Epoch    0 | Loss: 17.1256 | Accuracy: 70.0%
  Epoch   10 | Loss: 17.5229 | Accuracy: 70.0%
  Epoch   20 | Loss: 11.0442 | Accuracy: 83.8%
  Epoch   30 | Loss: 4.8296 | Accuracy: 95.0%
  Epoch   40 | Loss: 3.4505 | Accuracy: 97.5%
  Epoch   50 | Loss: 2.5040 | Accuracy: 98.8%
  Epoch   60 | Loss: 1.8125 | Accuracy: 97.5%
  Epoch   70 | Loss: 1.7134 | Accuracy: 98.8%
  Epoch   80 | Loss: 1.3676 | Accuracy: 100.0%
  100% alcanzado! Deteniendo entrenamiento.

Resultados en puntos de prueba:
  [0.0, 0.0] -> 0.9959 (dentro, esperado dentro) OK
  [0.5, 0.5] -> 0.6674 (dentro, esperado dentro) OK
  [1.2, 1.2] -> 0.0024 (fuera, esperado fuera) OK
  [0.0, 1.2] -> 0.0312 (fuera, esperado fuera) OK
  [-0.3, 0.3] -> 0.9920 (dentro, esperado dentro) OK
